In [174]:
"""
Improved Modeling Pipeline — WV Opioid Data
Target: LA_Opioid_Rate (next year)
Improvements:
  - rate_lag1 re-enabled
  - Trend & rolling features added
  - Interaction term (poverty × rurality)
  - ElasticNetCV + XGBoost added
  - Leave-one-year-out CV for reliable evaluation
  - Bootstrap augmentation on training data
  - RUCC excluded from scaling
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils import resample
from xgboost import XGBRegressor

In [175]:
# ── 1. Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv("../data/west_virginia_opioid_data_clean.csv") 
df

,Year,FIPS_Code,State,County,Labor_Force_Participation_Rate,Unemployment_Rate,Median_Household_Income,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Pct_Never_Married,Pct_Divorced,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Rural_Urban_Continuum_Code,LA_Opioid_Rate,LA_Opioid_Prscrbng_Rate_1Y_Chg
0,2013,54001,West Virginia,Barbour,51.8,8.0,37327,17.6,28.7,55.1,7.4,14.6,46.1,11.7,6.0,5.85,4.710000
1,2014,54001,West Virginia,Barbour,49.7,8.9,36351,19.8,29.1,53.5,7.7,12.8,48.2,12.2,6.0,10.56,4.710000
2,2015,54001,West Virginia,Barbour,51.2,9.0,37066,21.5,30.3,52.0,7.4,12.7,47.1,11.9,6.0,10.95,0.390000
3,2016,54001,West Virginia,Barbour,51.5,7.9,36733,22.4,32.5,51.1,7.1,13.2,43.9,12.8,6.0,7.85,-3.100000
4,2017,54001,West Virginia,Barbour,51.1,8.9,37516,22.5,32.5,49.6,7.6,12.1,44.5,15.3,6.0,7.47,-0.380000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,2019,54109,West Virginia,Wyoming,39.9,13.4,42332,22.4,29.4,59.7,8.4,11.4,48.7,9.3,6.0,0.39,-0.196667
601,2020,54109,West Virginia,Wyoming,41.0,11.6,44095,21.4,29.8,62.6,7.7,13.5,46.6,11.8,6.0,0.36,-0.023333
602,2021,54109,West Virginia,Wyoming,36.9,8.0,44630,25.3,31.2,55.8,8.0,14.0,46.7,11.6,6.0,0.51,0.150000
603,2022,54109,West Virginia,Wyoming,37.3,6.4,44510,24.4,31.3,51.7,9.4,15.8,46.6,11.4,6.0,0.94,0.430000


In [176]:
# ── 2. Sort ──────────────────────────────────────────────────────────────────
df_sorted = df.sort_values(['FIPS_Code', 'Year']).reset_index(drop=True)

In [177]:
# ── 3. Feature engineering ────────────────────────────────────────────────────
# Lag: previous year's opioid rate (fill first year with own rate)
df_sorted['rate_lag1'] = df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate'].shift(1)
df_sorted['rate_lag1'] = df_sorted.groupby('FIPS_Code')['rate_lag1'].transform(
    lambda x: x.fillna(df_sorted.loc[x.index, 'LA_Opioid_Rate'])
)

# Trend deltas: year-over-year change (fill NaN for first year with 0 — no change)
df_sorted['rate_delta1']    = df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate'].diff(1).fillna(0)
df_sorted['unemp_delta1']   = df_sorted.groupby('FIPS_Code')['Unemployment_Rate'].diff(1).fillna(0)
df_sorted['income_delta1']  = df_sorted.groupby('FIPS_Code')['Median_Household_Income'].diff(1).fillna(0)
df_sorted['poverty_delta1'] = df_sorted.groupby('FIPS_Code')['Poverty_Percent_All_Ages'].diff(1).fillna(0)
df_sorted['lfpr_delta1']    = df_sorted.groupby('FIPS_Code')['Labor_Force_Participation_Rate'].diff(1).fillna(0)

# Smoothed signal: 3-year rolling average of opioid rate
df_sorted['rate_rolling3'] = (
    df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# Interaction: poverty × rurality
df_sorted['poverty_x_rucc'] = (
    df_sorted['Poverty_Percent_All_Ages'] * df_sorted['Rural_Urban_Continuum_Code']
)

# Target: next year's opioid rate
df_sorted['target_next'] = df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate'].shift(-1)

# Drop rows with no target (2023 held separately)
df_model = df_sorted.dropna(subset=['target_next']).reset_index(drop=True)

print(f"Total rows before engineering:         {len(df)}")
print(f"Rows available for modeling (≤2022):   {len(df_model)}")
print(f"Rows for 2023 prediction:              {len(df_sorted[df_sorted['Year'] == 2023])}\n")

Total rows before engineering:         605
Rows available for modeling (≤2022):   550
Rows for 2023 prediction:              55



In [178]:
# ── 4. Features ───────────────────────────────────────────────────────────────
feature_cols = [
    'Labor_Force_Participation_Rate',
    'Unemployment_Rate',
    'Median_Household_Income',
    'Poverty_Percent_All_Ages',
    'Poverty_Percent_Age_0_17',
    'Pct_Never_Married',
    'Pct_Divorced',
    'Pct_Less_Than_HS',
    'Pct_HS_Grad',
    'Pct_Bachelors_Plus',
    'Rural_Urban_Continuum_Code',
    'rate_lag1',
    'rate_delta1',
    'unemp_delta1',
    'income_delta1',
    'poverty_delta1',
    'lfpr_delta1',
    'rate_rolling3',
    'poverty_x_rucc',
]

cols_to_scale = [c for c in feature_cols if c != 'Rural_Urban_Continuum_Code']
cols_no_scale = ['Rural_Urban_Continuum_Code']

X = df_model[feature_cols]
y = df_model['target_next']

In [179]:
X.isna().sum()

Labor_Force_Participation_Rate    0
Unemployment_Rate                 0
Median_Household_Income           0
Poverty_Percent_All_Ages          0
Poverty_Percent_Age_0_17          0
Pct_Never_Married                 0
Pct_Divorced                      0
Pct_Less_Than_HS                  0
Pct_HS_Grad                       0
Pct_Bachelors_Plus                0
Rural_Urban_Continuum_Code        0
rate_lag1                         0
rate_delta1                       0
unemp_delta1                      0
income_delta1                     0
poverty_delta1                    0
lfpr_delta1                       0
rate_rolling3                     0
poverty_x_rucc                    0
dtype: int64

In [ ]:
# ── 5. Leave-one-year-out evaluation ─────────────────────────────────────────
holdout_years = [2020, 2021, 2022]

model_configs = {
    'Ridge':             lambda: RidgeCV(alphas=np.logspace(-3, 6, 100), cv=TimeSeriesSplit(n_splits=2)),
    'ElasticNet':        lambda: ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0],
                                              alphas=np.logspace(-3, 3, 50),
                                              cv=TimeSeriesSplit(n_splits=2), max_iter=5000),
    'Random Forest':     lambda: RandomForestRegressor(n_estimators=500, max_depth=6,
                                                        min_samples_leaf=5, random_state=42),
    'Gradient Boosting': lambda: GradientBoostingRegressor(n_estimators=500, max_depth=4,
                                                            learning_rate=0.05, min_samples_leaf=5,
                                                            random_state=42),
    'XGBoost':           lambda: XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                              subsample=0.8, colsample_bytree=0.8,
                                              min_child_weight=5, random_state=42,
                                              verbosity=0),
}

loyo_results = {name: {'r2': [], 'rmse': []} for name in model_configs}

for test_year in holdout_years:
    train_mask = df_model['Year'] < test_year
    test_mask  = df_model['Year'] == test_year

    X_tr_raw, y_tr = X[train_mask].copy(), y[train_mask]
    X_te_raw, y_te = X[test_mask].copy(),  y[test_mask]

    # Bootstrap augmentation on training fold
    augmented = []
    for _, group in df_model[train_mask].groupby('FIPS_Code'):
        boot = resample(group, n_samples=len(group) * 2, random_state=42)
        augmented.append(boot)
    df_aug  = pd.concat(augmented).reset_index(drop=True)
    X_tr_aug = df_aug[feature_cols]
    y_tr_aug = df_aug['target_next']

    # Scale (fit on augmented train, exclude RUCC)
    scaler = StandardScaler()
    X_tr_scaled = np.hstack([scaler.fit_transform(X_tr_aug[cols_to_scale]),
                              X_tr_aug[cols_no_scale].values])
    X_te_scaled = np.hstack([scaler.transform(X_te_raw[cols_to_scale]),
                              X_te_raw[cols_no_scale].values])

    for name, build in model_configs.items():
        model = build()
        is_linear = name in ('Ridge', 'ElasticNet')

        model.fit(X_tr_scaled if is_linear else X_tr_aug, y_tr_aug)
        preds = model.predict(X_te_scaled if is_linear else X_te_raw)

        loyo_results[name]['r2'].append(r2_score(y_te, preds))
        loyo_results[name]['rmse'].append(root_mean_squared_error(y_te, preds))

In [ ]:
# ── 6. Print LOYO summary ─────────────────────────────────────────────────────
print("Leave-One-Year-Out CV Results (mean ± std across 2020/2021/2022)")
print("=" * 58)
print(f"{'Model':<22} {'R² mean':>9} {'R² std':>8} {'RMSE mean':>10}")
print("=" * 58)

for name, res in loyo_results.items():
    print(f"{name:<22} {np.mean(res['r2']):>9.4f} {np.std(res['r2']):>8.4f} {np.mean(res['rmse']):>10.4f}")

print("=" * 58)

Leave-One-Year-Out CV Results (mean ± std across 2020/2021/2022)
Model                    R² mean   R² std  RMSE mean
Ridge                     0.8645   0.0531     1.6741
ElasticNet                0.8575   0.0707     1.7128
Random Forest             0.8468   0.0496     1.8025
Gradient Boosting         0.8518   0.0517     1.7583
XGBoost                   0.8578   0.0622     1.7136


In [182]:
# ── 7. Final model: retrain on all data ≤ 2022, evaluate on 2022 ──────────────
train_mask = df_model['Year'] <= 2021
test_mask  = df_model['Year'] == 2022

# Bootstrap augment final training set
augmented = []
for _, group in df_model[train_mask].groupby('FIPS_Code'):
    boot = resample(group, n_samples=len(group) * 2, random_state=42)
    augmented.append(boot)
df_aug_final = pd.concat(augmented).reset_index(drop=True)

X_train_aug = df_aug_final[feature_cols]
y_train_aug = df_aug_final['target_next']
X_test_raw  = X[test_mask]
y_test      = y[test_mask]

scaler_final = StandardScaler()
X_train_scaled = np.hstack([scaler_final.fit_transform(X_train_aug[cols_to_scale]),
                             X_train_aug[cols_no_scale].values])
X_test_scaled  = np.hstack([scaler_final.transform(X_test_raw[cols_to_scale]),
                             X_test_raw[cols_no_scale].values])

final_models = {name: build() for name, build in model_configs.items()}
final_results = {}

print("\nFinal Evaluation on 2022 Holdout")
print("=" * 45)
print(f"{'Model':<22} {'R²':>7} {'RMSE':>8} {'MSE':>10}")
print("=" * 45)

for name, model in final_models.items():
    is_linear = name in ('Ridge', 'ElasticNet')
    model.fit(X_train_scaled if is_linear else X_train_aug, y_train_aug)
    preds = model.predict(X_test_scaled if is_linear else X_test_raw)
    r2   = r2_score(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    mse  = mean_squared_error(y_test, preds)
    final_results[name] = {'model': model, 'preds': preds, 'r2': r2, 'rmse': rmse, 'mse': mse}
    print(f"{name:<22} {r2:>7.4f} {rmse:>8.4f} {mse:>10.4f}")

print("=" * 45)


Final Evaluation on 2022 Holdout
Model                       R²     RMSE        MSE
Ridge                   0.8920   1.2949     1.6767
ElasticNet              0.9145   1.1518     1.3267
Random Forest           0.9100   1.1822     1.3976
Gradient Boosting       0.8831   1.3470     1.8144
XGBoost                 0.8990   1.2524     1.5684


In [183]:
# ── 8. Best model ─────────────────────────────────────────────────────────────
best_name = max(final_results, key=lambda k: final_results[k]['r2'])
print(f"\nBest model: {best_name} (R²={final_results[best_name]['r2']:.4f})")



Best model: ElasticNet (R²=0.9145)


In [184]:
# ── 9. Feature importance ─────────────────────────────────────────────────────
best_model = final_results[best_name]['model']
print("\nFeature Importance:")

if best_name in ('Ridge', 'ElasticNet'):
    importance = pd.DataFrame({
        'Feature':    feature_cols,
        'Importance': np.abs(best_model.coef_)
    })
else:
    importance = pd.DataFrame({
        'Feature':    feature_cols,
        'Importance': best_model.feature_importances_
    })

importance = importance.sort_values('Importance', ascending=False)
print(importance.to_string(index=False))


Feature Importance:
                       Feature  Importance
                   lfpr_delta1    1.777695
    Rural_Urban_Continuum_Code    1.620808
                     rate_lag1    0.878746
Labor_Force_Participation_Rate    0.344902
       Median_Household_Income    0.252598
                  unemp_delta1    0.162468
             Unemployment_Rate    0.140535
                   rate_delta1    0.124737
                poverty_x_rucc    0.098121
                  Pct_Divorced    0.092100
             Pct_Never_Married    0.071293
                poverty_delta1    0.068875
                 income_delta1    0.058975
              Pct_Less_Than_HS    0.058294
                   Pct_HS_Grad    0.022116
            Pct_Bachelors_Plus    0.000000
      Poverty_Percent_All_Ages    0.000000
      Poverty_Percent_Age_0_17    0.000000
                 rate_rolling3    0.000000


In [185]:
# ── 10. Predict 2024 from 2023 data ──────────────────────────────────────────
rate_2022 = df_sorted[df_sorted['Year'] == 2022].set_index('FIPS_Code')['LA_Opioid_Rate']
pred_rows = df_sorted[df_sorted['Year'] == 2023].copy().reset_index(drop=True)
pred_rows['rate_lag1'] = pred_rows['FIPS_Code'].map(rate_2022)

X_pred = pred_rows[feature_cols].fillna(pred_rows[feature_cols].median())

is_linear_best = best_name in ('Ridge', 'ElasticNet')
if is_linear_best:
    X_pred_eval = np.hstack([scaler_final.transform(X_pred[cols_to_scale]),
                              X_pred[cols_no_scale].values])
else:
    X_pred_eval = X_pred

pred_rows['predicted_rate'] = best_model.predict(X_pred_eval)

# Distress score
pred_rows['distress_score'] = (
    pred_rows['predicted_rate'].rank(pct=True) +
    (1 - pred_rows['Labor_Force_Participation_Rate'].rank(pct=True)) +
    (1 - pred_rows['Median_Household_Income'].rank(pct=True)) +
    pred_rows['Unemployment_Rate'].rank(pct=True) +
    pred_rows['Poverty_Percent_All_Ages'].rank(pct=True)
)

top10 = (
    pred_rows[['FIPS_Code', 'State', 'County', 'predicted_rate',
               'Labor_Force_Participation_Rate', 'Median_Household_Income',
               'Poverty_Percent_All_Ages', 'Unemployment_Rate', 'distress_score']]
    .sort_values('distress_score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1

print(f"\nTop 10 Counties for Immediate Clinic Intervention (via {best_name}):")
print(top10.to_string())


Top 10 Counties for Immediate Clinic Intervention (via ElasticNet):
    FIPS_Code          State    County  predicted_rate  Labor_Force_Participation_Rate  Median_Household_Income  Poverty_Percent_All_Ages  Unemployment_Rate  distress_score
1       54013  West Virginia   Calhoun        5.257632                            40.2                    41421                      33.4               10.9        4.545455
2       54047  West Virginia  McDowell        1.535657                            28.5                    29980                      30.9               14.9        4.145455
3       54059  West Virginia     Mingo        1.769216                            36.7                    39527                      29.9               10.8        4.036364
4       54101  West Virginia   Webster        2.993812                            41.0                    42061                      22.1                9.6        3.981818
5       54015  West Virginia      Clay        2.019954            